In [1]:
import sys
import os
sys.path.append(os.path.abspath(".."))

from spark_config import SparkConfig
from logger import LoggerConfig
from io_config import IOConfig
from platform_app import PlatformApp

In [2]:
logger = LoggerConfig().setup_logger(log_dir=IOConfig.LOG_DIR)
spark = SparkConfig.create_spark(app_name="SQL in Spark", logger=logger, use_databricks=True)

2026-03-05 14:44:26 | INFO     | Logging configured: level=DEBUG, format=colored
2026-03-05 14:44:27 | INFO     | Connected to Databricks via Spark Connect.


In [3]:
df = (
        spark.read.
        format("csv").
        option("header", True).
        option("inferSchema", True).
        csv("/Volumes/databricks_toturial/source/source_data/movies")
    )
df.show(n=10, truncate=False)

+-------------------------------------------+---------+------------+-----------+-------------------------+-------+--------+---------+--------+--------+
|title                                      |industry |release_year|imdb_rating|studio                   |budget |revenue |unit     |currency|language|
+-------------------------------------------+---------+------------+-----------+-------------------------+-------+--------+---------+--------+--------+
|Pather Panchali                            |Bollywood|1955        |8.3        |Government of West Bengal|70000.0|100000.0|Thousands|INR     |Bengali |
|Doctor Strange in the Multiverse of Madness|Hollywood|2022        |7          |Marvel Studios           |200.0  |954.8   |Millions |USD     |English |
|Thor: The Dark World                       |Hollywood|2013        |6.8        |Marvel Studios           |165.0  |644.8   |Millions |USD     |English |
|Thor: Ragnarok                             |Hollywood|2017        |7.9        |Marvel S

In [4]:
from pyspark.sql import functions as F

df_narrow = df.select("title", "studio", "imdb_rating").filter(F.col("release_year") >= 2010)
df_narrow.explain("extended")

== Parsed Logical Plan ==
'Filter '`>=`('release_year, 2010)
+- 'Project ['title, 'studio, 'imdb_rating]
   +- Relation [title#13430,industry#13431,release_year#13432,imdb_rating#13433,studio#13434,budget#13435,revenue#13436,unit#13437,currency#13438,language#13439] csv

== Analyzed Logical Plan ==
title: string, studio: string, imdb_rating: string
Project [title#13430, studio#13434, imdb_rating#13433]
+- Filter (release_year#13432 >= 2010)
   +- Project [title#13430, studio#13434, imdb_rating#13433, release_year#13432]
      +- Relation [title#13430,industry#13431,release_year#13432,imdb_rating#13433,studio#13434,budget#13435,revenue#13436,unit#13437,currency#13438,language#13439] csv

== Optimized Logical Plan ==
Project [title#13430, studio#13434, imdb_rating#13433]
+- Filter (isnotnull(release_year#13432) AND (release_year#13432 >= 2010))
   +- Relation [title#13430,industry#13431,release_year#13432,imdb_rating#13433,studio#13434,budget#13435,revenue#13436,unit#13437,currency#13438

### Print only Physical Plan

In [5]:
df_narrow.explain("formatted")

== Physical Plan ==
PhotonResultStage (6)
+- PhotonColumnarToRow (5)
   +- PhotonProject (4)
      +- PhotonFilter (3)
         +- PhotonRowToColumnar (2)
            +- Scan csv  (1)


(1) Scan csv 
Output [4]: [title#13430, release_year#13432, imdb_rating#13433, studio#13434]
Batched: false
Location: InMemoryFileIndex [dbfs:/Volumes/databricks_toturial/source/source_data/movies]
PushedFilters: [IsNotNull(release_year), GreaterThanOrEqual(release_year,2010)]
ReadSchema: struct<title:string,release_year:int,imdb_rating:string,studio:string>

(2) PhotonRowToColumnar
Input [4]: [title#13430, release_year#13432, imdb_rating#13433, studio#13434]

(3) PhotonFilter
Input [4]: [title#13430, release_year#13432, imdb_rating#13433, studio#13434]
Arguments: (isnotnull(release_year#13432) AND (release_year#13432 >= 2010))

(4) PhotonProject
Input [4]: [title#13430, release_year#13432, imdb_rating#13433, studio#13434]
Arguments: [title#13430, studio#13434, imdb_rating#13433]

(5) PhotonColumnarToRo

In [6]:
df_narrow.show(truncate=False)

+-------------------------------------------+---------------------+-----------+
|title                                      |studio               |imdb_rating|
+-------------------------------------------+---------------------+-----------+
|Doctor Strange in the Multiverse of Madness|Marvel Studios       |7          |
|Thor: The Dark World                       |Marvel Studios       |6.8        |
|Thor: Ragnarok                             |Marvel Studios       |7.9        |
|Thor: Love and Thunder                     |Marvel Studios       |6.8        |
|Interstellar                               |Warner Bros. Pictures|8.6        |
|Parasite                                   |NULL                 |8.5        |
|Avengers: Endgame                          |Marvel Studios       |8.4        |
|Avengers: Infinity War                     |Marvel Studios       |8.4        |
|Captain America: The First Avenger         |Marvel Studios       |6.9        |
|Captain America: The Winter Soldier    